# 📈 Linear Regression Deep Dive

> **Master linear regression from theory to implementation**

This notebook provides a comprehensive exploration of linear regression, from mathematical foundations to practical implementation. You'll build everything from scratch and compare with scikit-learn.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Understand** the mathematical foundation of linear regression
- **Implement** linear regression from scratch using NumPy
- **Master** feature engineering and data preprocessing
- **Evaluate** model performance using multiple metrics
- **Compare** different optimization approaches
- **Apply** regularization techniques (Ridge, Lasso)

## 📊 Dataset: Boston Housing (Synthetic)

We'll use a synthetic version of the Boston Housing dataset to predict house prices based on various features.

**Features:**
- `CRIM`: Crime rate per capita
- `ZN`: Proportion of residential land zoned for lots over 25,000 sq.ft
- `INDUS`: Proportion of non-retail business acres
- `CHAS`: Charles River dummy variable
- `NOX`: Nitric oxides concentration
- `RM`: Average number of rooms per dwelling
- `AGE`: Proportion of owner-occupied units built prior to 1940
- `DIS`: Weighted distances to employment centers
- `RAD`: Index of accessibility to radial highways
- `TAX`: Property tax rate
- `PTRATIO`: Pupil-teacher ratio
- `B`: Proportion of blacks by town
- `LSTAT`: % lower status of the population

**Target:** `MEDV` - Median value of homes in $1000s

## 🛠️ Setup and Imports

In [ ]:
# Essential imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All imports successful!")

## 📊 Data Generation and Exploration

In [ ]:
# Generate synthetic housing data
X, y = make_regression(
    n_samples=506,
    n_features=13,
    noise=10,
    random_state=42
)

# Create feature names
feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE',
    'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
]

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['MEDV'] = y

# Scale target to realistic house prices (in thousands)
df['MEDV'] = (df['MEDV'] - df['MEDV'].min()) / (df['MEDV'].max() - df['MEDV'].min()) * 40 + 10

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(feature_names)}")
print(f"Target range: ${df['MEDV'].min():.1f}k - ${df['MEDV'].max():.1f}k")

# Display first few rows
df.head()

In [ ]:
# Comprehensive data exploration
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Target distribution
axes[0, 0].hist(df['MEDV'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of House Prices')
axes[0, 0].set_xlabel('Price ($1000s)')
axes[0, 0].set_ylabel('Frequency')

# Correlation heatmap
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, ax=axes[0, 1], cbar_kws={"shrink": .8})
axes[0, 1].set_title('Feature Correlation Matrix')

# Feature vs target scatter (strongest correlation)
strongest_corr_feature = corr_matrix['MEDV'].abs().sort_values(ascending=False).index[1]
axes[1, 0].scatter(df[strongest_corr_feature], df['MEDV'], alpha=0.6)
axes[1, 0].set_xlabel(strongest_corr_feature)
axes[1, 0].set_ylabel('House Price ($1000s)')
axes[1, 0].set_title(f'Price vs {strongest_corr_feature} (Strongest Correlation)')

# Box plot of target
axes[1, 1].boxplot(df['MEDV'])
axes[1, 1].set_ylabel('House Price ($1000s)')
axes[1, 1].set_title('House Price Distribution (Box Plot)')

plt.tight_layout()
plt.show()

# Summary statistics
print("\n📊 Summary Statistics:")
print(df.describe().round(2))

## 🧮 Mathematical Foundation

### Linear Regression Equation

The linear regression model can be expressed as:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

In matrix form:
$$\hat{y} = X\beta$$

### Normal Equation

The optimal parameters can be found using:
$$\beta = (X^T X)^{-1} X^T y$$

### Cost Function (Mean Squared Error)

$$J(\beta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\beta(x^{(i)}) - y^{(i)})^2$$

## 🔨 Linear Regression from Scratch

In [ ]:
class LinearRegressionFromScratch:
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.weights = None
        self.bias = None
        self.cost_history = []
        
    def add_bias_term(self, X):
        """Add bias term (column of ones) to feature matrix"""
        return np.column_stack([np.ones(X.shape[0]), X])
    
    def compute_cost(self, X, y, weights):
        """Compute mean squared error cost"""
        m = X.shape[0]
        predictions = X @ weights
        cost = (1 / (2 * m)) * np.sum((predictions - y) ** 2)
        return cost
    
    def fit_normal_equation(self, X, y):
        """Fit using normal equation (analytical solution)"""
        X_with_bias = self.add_bias_term(X)
        
        # Normal equation: θ = (X^T X)^(-1) X^T y
        try:
            self.weights = np.linalg.inv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
        except np.linalg.LinAlgError:
            # Use pseudo-inverse if matrix is singular
            self.weights = np.linalg.pinv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
        
        self.bias = self.weights[0]
        self.weights = self.weights[1:]
        
        # Calculate final cost
        final_cost = self.compute_cost(X_with_bias, y, 
                                     np.concatenate([[self.bias], self.weights]))
        self.cost_history = [final_cost]
        
        return self
    
    def fit_gradient_descent(self, X, y):
        """Fit using gradient descent"""
        m, n = X.shape
        
        # Initialize parameters
        self.weights = np.random.normal(0, 0.01, n)
        self.bias = 0
        self.cost_history = []
        
        for i in range(self.max_iterations):
            # Forward pass
            predictions = X @ self.weights + self.bias
            
            # Compute cost
            cost = (1 / (2 * m)) * np.sum((predictions - y) ** 2)
            self.cost_history.append(cost)
            
            # Compute gradients
            dw = (1 / m) * X.T @ (predictions - y)
            db = (1 / m) * np.sum(predictions - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Check for convergence
            if i > 0 and abs(self.cost_history[-2] - self.cost_history[-1]) < self.tolerance:
                print(f"Converged after {i+1} iterations")
                break
        
        return self
    
    def predict(self, X):
        """Make predictions"""
        if self.weights is None:
            raise ValueError("Model not fitted yet. Call fit() first.")
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        """Calculate R² score"""
        predictions = self.predict(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)
    
    def plot_cost_history(self):
        """Plot cost function over iterations"""
        if len(self.cost_history) > 1:
            plt.figure(figsize=(10, 6))
            plt.plot(self.cost_history)
            plt.title('Cost Function Over Iterations')
            plt.xlabel('Iteration')
            plt.ylabel('Cost (MSE)')
            plt.grid(True)
            plt.show()
        else:
            print("Cost history not available (normal equation used)")

print("✅ LinearRegressionFromScratch class defined!")

## 🔄 Data Preprocessing and Model Training

In [ ]:
# Prepare features and target
X = df[feature_names].values
y = df['MEDV'].values

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Target range - Train: [{y_train.min():.1f}, {y_train.max():.1f}]")
print(f"Target range - Test: [{y_test.min():.1f}, {y_test.max():.1f}]")

In [ ]:
# Train models using different approaches
print("🔨 Training Linear Regression Models...\n")

# 1. Our implementation with Normal Equation
print("1️⃣ Training with Normal Equation...")
lr_normal = LinearRegressionFromScratch()
lr_normal.fit_normal_equation(X_train_scaled, y_train)
print(f"   Final cost: {lr_normal.cost_history[-1]:.4f}")

# 2. Our implementation with Gradient Descent
print("\n2️⃣ Training with Gradient Descent...")
lr_gd = LinearRegressionFromScratch(learning_rate=0.01, max_iterations=1000)
lr_gd.fit_gradient_descent(X_train_scaled, y_train)
print(f"   Final cost: {lr_gd.cost_history[-1]:.4f}")

# 3. Scikit-learn implementation
print("\n3️⃣ Training with Scikit-learn...")
lr_sklearn = LinearRegression()
lr_sklearn.fit(X_train_scaled, y_train)
sklearn_predictions = lr_sklearn.predict(X_train_scaled)
sklearn_cost = mean_squared_error(y_train, sklearn_predictions) / 2
print(f"   Training cost: {sklearn_cost:.4f}")

print("\n✅ All models trained successfully!")

In [ ]:
# Visualize gradient descent convergence
lr_gd.plot_cost_history()

## 📊 Model Evaluation and Comparison

In [ ]:
# Make predictions on test set
pred_normal = lr_normal.predict(X_test_scaled)
pred_gd = lr_gd.predict(X_test_scaled)
pred_sklearn = lr_sklearn.predict(X_test_scaled)

# Calculate metrics for all models
def calculate_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

# Compare all models
results = [
    calculate_metrics(y_test, pred_normal, 'Normal Equation'),
    calculate_metrics(y_test, pred_gd, 'Gradient Descent'),
    calculate_metrics(y_test, pred_sklearn, 'Scikit-learn')
]

results_df = pd.DataFrame(results)
print("📊 Model Comparison Results:")
print(results_df.round(4))

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ('Normal Equation', pred_normal),
    ('Gradient Descent', pred_gd),
    ('Scikit-learn', pred_sklearn)
]

for i, (name, predictions) in enumerate(models):
    axes[i].scatter(y_test, predictions, alpha=0.6)
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[i].set_xlabel('Actual Prices')
    axes[i].set_ylabel('Predicted Prices')
    axes[i].set_title(f'{name}\nR² = {r2_score(y_test, predictions):.4f}')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 Practice Problems

Now it's your turn to practice! Complete the following exercises:

### **Problem 1: Feature Importance Analysis**
Analyze which features are most important for predicting house prices. Create a visualization showing feature coefficients.

In [ ]:
# Your code here
# Hint: Use the weights from your trained model
# Create a bar plot showing feature importance

# Solution template:
def analyze_feature_importance(model, feature_names):
    """
    Analyze and visualize feature importance
    
    Parameters:
    model: trained linear regression model
    feature_names: list of feature names
    """
    # TODO: Extract coefficients
    # TODO: Create visualization
    # TODO: Interpret results
    pass

# Call your function
# analyze_feature_importance(lr_normal, feature_names)

### **Problem 2: Regularization Implementation**
Implement Ridge regression from scratch and compare it with regular linear regression.

In [ ]:
# Your code here
# Implement Ridge regression with L2 regularization

class RidgeRegressionFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Regularization strength
        self.weights = None
        self.bias = None
    
    def fit(self, X, y):
        """
        Fit Ridge regression using normal equation with regularization
        θ = (X^T X + αI)^(-1) X^T y
        """
        # TODO: Implement Ridge regression
        # Hint: Add alpha * I to X^T X before inversion
        pass
    
    def predict(self, X):
        # TODO: Implement prediction
        pass

# Test your implementation
# ridge_model = RidgeRegressionFromScratch(alpha=1.0)
# ridge_model.fit(X_train_scaled, y_train)
# ridge_predictions = ridge_model.predict(X_test_scaled)

### **Problem 3: Learning Rate Experiment**
Experiment with different learning rates and visualize how they affect convergence.

In [ ]:
# Your code here
# Test learning rates: [0.001, 0.01, 0.1, 1.0]
# Plot cost history for each learning rate

learning_rates = [0.001, 0.01, 0.1, 1.0]

# TODO: Train models with different learning rates
# TODO: Plot cost histories on the same graph
# TODO: Analyze which learning rate works best

## 🎯 Key Takeaways

From this notebook, you should understand:

1. **Mathematical Foundation**: Linear regression finds the best linear relationship between features and target
2. **Implementation Methods**: Both analytical (normal equation) and iterative (gradient descent) approaches work
3. **Feature Scaling**: Essential for gradient descent to converge properly
4. **Model Evaluation**: Multiple metrics (MSE, RMSE, MAE, R²) provide different insights
5. **Regularization**: Helps prevent overfitting in complex models

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Try polynomial features** to capture non-linear relationships
3. **Experiment with different datasets** from scikit-learn
4. **Move to the next notebook**: Logistic Regression Deep Dive

## 📚 Additional Resources

- [Scikit-learn Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
- [Andrew Ng's ML Course](https://www.coursera.org/learn/machine-learning)
- [Elements of Statistical Learning](https://web.stanford.edu/~hastie/ElemStatLearn/)

---

**Great job completing this notebook!** 🎉

You've built linear regression from scratch and understand both the theory and implementation. This foundation will serve you well as you progress to more advanced algorithms.